In [3]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
from tqdm import tqdm
import numpy as np

# ================= 配置区域 =================
DEVICE = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

# [修改] 对应新的 3通道 数据集
DATASET_ROOT = "./dataset_tri_RAD_Final_Fixed"
SIM_TRAIN_DIR = os.path.join(DATASET_ROOT, "sim_train")
SIM_VAL_DIR = os.path.join(DATASET_ROOT, "sim_val")
REAL_TEST_DIR = os.path.join(DATASET_ROOT, "real_test")

RESULTS_DIR = "./training_results_RAD_Final" 
CLASS_NAMES = [
    "BPSK", "QPSK", "8PSK", "OQPSK", "PI4DQPSK", 
    "16QAM", "32QAM", "64QAM", 
    "2ASK", "16APSK", "32APSK", "FSK2"
]
NUM_CLASSES = len(CLASS_NAMES)
IMAGE_SIZE = 64
BATCH_SIZE = 128
LEARNING_RATE = 0.001
NUM_EPOCHS = 100

os.makedirs(RESULTS_DIR, exist_ok=True)
BEST_MODEL_PATH = os.path.join(RESULTS_DIR, "best_model_rad.pth")

# ================= 1. 数据集 (不变) =================
class InMemoryDataset(Dataset):
    def __init__(self, root_dir, has_snr=False):
        self.data = []
        self.labels = []
        
        if not os.path.exists(root_dir): return
        print(f"Loading {root_dir}...")
        files = []
        for cls_name in CLASS_NAMES:
            d = Path(root_dir) / cls_name
            if d.exists(): 
                found = list(d.glob("*.pt"))
                files.extend([(p, CLASS_NAMES.index(cls_name)) for p in found])
        
        with tqdm(total=len(files), unit="file") as pbar:
            for p, label in files:
                try:
                    tensor = torch.load(p) # (3, 64, 64)
                    self.data.append(tensor) 
                    self.labels.append(label)
                except: pass
                pbar.update(1)

    def __len__(self): return len(self.data)
    def __getitem__(self, idx): return self.data[idx], self.labels[idx], 0.0

# ================= 2. 模型 (改为 3 通道输入) =================
class TinyCNN(nn.Module):
    def __init__(self, num_classes):
        super(TinyCNN, self).__init__()
        self.features = nn.Sequential(
            # [修改] 输入 3 通道 (R, A, D)
            nn.Conv2d(3, 16, 3, padding=1), 
            nn.BatchNorm2d(16), nn.ReLU(), nn.MaxPool2d(2),
            nn.Dropout2d(0.1),
            
            nn.Conv2d(16, 32, 3, padding=1), 
            nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Dropout2d(0.1),
            
            nn.Conv2d(32, 64, 3, padding=1), 
            nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64*8*8, 128),
            nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(128, num_classes)
        )
    def forward(self, x): return self.classifier(self.features(x))

# ================= 3. 主程序 =================
def main():
    print("=== Training 3-Channel RAD Model ===")
    
    train_ds = InMemoryDataset(SIM_TRAIN_DIR) 
    val_ds = InMemoryDataset(SIM_VAL_DIR) 
    real_ds = InMemoryDataset(REAL_TEST_DIR)
    
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
    
    model = TinyCNN(NUM_CLASSES).to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
    criterion = nn.CrossEntropyLoss()
    
    best_acc = 0.0
    for epoch in range(NUM_EPOCHS):
        model.train()
        correct, total = 0, 0
        for imgs, labels, _ in train_loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            _, preds = outputs.max(1)
            correct += preds.eq(labels).sum().item()
            total += labels.size(0)
        
        train_acc = correct / total
        
        model.eval()
        v_correct, v_total = 0, 0
        with torch.no_grad():
            for imgs, labels, _ in val_loader:
                imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
                out = model(imgs)
                _, preds = out.max(1)
                v_correct += preds.eq(labels).sum().item()
                v_total += labels.size(0)
        
        val_acc = v_correct / v_total
        print(f"Epoch {epoch+1} | Train: {train_acc:.4f} | Val: {val_acc:.4f}")
        
        if val_acc > best_acc:
            best_acc = val_acc
            torch.save(model.state_dict(), BEST_MODEL_PATH)
    
    # Test Real
    print("\nTesting Real Data...")
    model.load_state_dict(torch.load(BEST_MODEL_PATH))
    model.eval()
    if len(real_ds) > 0:
        real_loader = DataLoader(real_ds, batch_size=BATCH_SIZE)
        y_true, y_pred = [], []
        with torch.no_grad():
            for imgs, labels, _ in real_loader:
                imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
                _, preds = model(imgs).max(1)
                y_true.extend(labels.cpu().numpy())
                y_pred.extend(preds.cpu().numpy())
        
        acc = np.mean(np.array(y_true) == np.array(y_pred))
        print(f"Real Data Accuracy: {acc:.4f}")

if __name__ == "__main__":
    main()

=== Training 3-Channel RAD Model ===
Loading ./dataset_tri_RAD_Final_Fixed\sim_train...


100%|██████████| 7642/7642 [00:00<00:00, 9376.69file/s]


Loading ./dataset_tri_RAD_Final_Fixed\sim_val...


100%|██████████| 1958/1958 [00:00<00:00, 8024.63file/s]


Loading ./dataset_tri_RAD_Final_Fixed\real_test...


100%|██████████| 4500/4500 [00:00<00:00, 10044.65file/s]


Epoch 1 | Train: 0.2350 | Val: 0.5787
Epoch 2 | Train: 0.4614 | Val: 0.7380
Epoch 3 | Train: 0.5877 | Val: 0.7743
Epoch 4 | Train: 0.6256 | Val: 0.7722
Epoch 5 | Train: 0.6513 | Val: 0.8039
Epoch 6 | Train: 0.6755 | Val: 0.8080
Epoch 7 | Train: 0.6820 | Val: 0.7957
Epoch 8 | Train: 0.6897 | Val: 0.8320
Epoch 9 | Train: 0.7022 | Val: 0.8264
Epoch 10 | Train: 0.7142 | Val: 0.8320
Epoch 11 | Train: 0.7167 | Val: 0.8330
Epoch 12 | Train: 0.7206 | Val: 0.8391
Epoch 13 | Train: 0.7252 | Val: 0.8401
Epoch 14 | Train: 0.7342 | Val: 0.8248
Epoch 15 | Train: 0.7393 | Val: 0.8514
Epoch 16 | Train: 0.7332 | Val: 0.8264
Epoch 17 | Train: 0.7374 | Val: 0.8315
Epoch 18 | Train: 0.7498 | Val: 0.8493
Epoch 19 | Train: 0.7511 | Val: 0.8371
Epoch 20 | Train: 0.7489 | Val: 0.8396
Epoch 21 | Train: 0.7518 | Val: 0.8391
Epoch 22 | Train: 0.7502 | Val: 0.8417
Epoch 23 | Train: 0.7659 | Val: 0.8437
Epoch 24 | Train: 0.7580 | Val: 0.8432
Epoch 25 | Train: 0.7660 | Val: 0.8483
Epoch 26 | Train: 0.7629 | Val: 0.